In [1]:
import os

os.environ['HADOOP_USER_NAME'] = 'hdfs'

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Jupyter_HDFS_Native_Read") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://master:9000") \
    .getOrCreate()

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
print("fs.defaultFS:", hadoop_conf.get("fs.defaultFS"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/14 10:24:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


fs.defaultFS: hdfs://master:9000


In [3]:
import sys
sys.path.append("src")

In [4]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DoubleType, TimestampType
from pathlib import Path
import json

TYPE_MAP = {
    "integer": IntegerType(),
    "string": StringType(),
    "float": FloatType(),
    "double": DoubleType(),
    "timestamp": TimestampType()
}

def load_schema_spark(table: str) -> StructType:
    schema_path = Path("config/schemas") / f"{table}_schema.json"
    with open(schema_path) as f:
        schema_def = json.load(f)
    fields = []
    for col in schema_def["columns"]:
        spark_type = TYPE_MAP.get(col["type"], StringType())
        fields.append(StructField(col["name"], spark_type, col["nullable"]))
    return StructType(fields)

In [5]:
table_path = "hdfs://master:9000/data/bronze/distribution_centers/distribution_centers.csv"

schema = load_schema_spark("distribution_centers")

df_cen = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv(table_path)

df_cen.show(20, truncate=False)

+---+-------------------------------------------+--------+---------+
|id |name                                       |latitude|longitude|
+---+-------------------------------------------+--------+---------+
|1  |Memphis TN                                 |35.1174 |-89.9711 |
|2  |Chicago IL                                 |41.8369 |-87.6847 |
|3  |Houston TX                                 |29.7604 |-95.3698 |
|4  |Los Angeles CA                             |34.05   |-118.25  |
|5  |New Orleans LA                             |29.95   |-90.0667 |
|6  |Port Authority of New York/New Jersey NY/NJ|40.634  |-73.7834 |
|7  |Philadelphia PA                            |39.95   |-75.1667 |
|8  |Mobile AL                                  |30.6944 |-88.0431 |
|9  |Charleston SC                              |32.7833 |-79.9333 |
|10 |Savannah GA                                |32.0167 |-81.1167 |
+---+-------------------------------------------+--------+---------+



In [6]:
df.count()

NameError: name 'df' is not defined

In [ ]:
len(df.columns)

In [ ]:
df.printSchema()

In [ ]:
from pyspark.sql.functions import col
df.filter(col("id").isNull()).count()

In [ ]:
df.groupby("id")\
    .count()\
    .filter(col("count") > 1)\
    .show()

In [ ]:
df.filter(col("id").isNotNull()).show()

In [ ]:
df.select("name").distinct().show(truncate = False)

In [ ]:
df.groupby("name").count().filter(col("count")>1) .show()

In [ ]:
from pyspark.sql.functions import col, trim

space_issues = df.filter(col("name") != trim(col("name")))

space_issues.show(truncate=False)


In [ ]:
table_path = "hdfs://master:9000/data/bronze/products/products.csv"

schema = load_schema_spark("products")

df_pro = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv(table_path)

# بديل لـ df.show()
df_pro.limit(5).toPandas()

In [ ]:
df.count()

In [ ]:
df.printSchema()

In [ ]:
df.filter(col("id").isNull()).count()

In [ ]:
df.groupby("id").count().filter(col("count")>1).show()


In [ ]:
df_cost = df.filter((col("cost") <= 0) | (col("cost").isNull()))
df_cost.show()

In [ ]:
df.select("category").distinct().show(50, truncate=False)

In [ ]:
from pyspark.sql import functions as F

# الصح: دالة trim بتاكل العمود جواها الفانكشن نفسها
df_spaces = df.filter(F.col("category") != F.trim(F.col("category")))

df_spaces.show(truncate=False)


In [ ]:
df.select("name").distinct().count()

In [ ]:
df.count()

In [ ]:
df.filter(F.col("name") != F.trim(F.col("name"))).show()

In [ ]:
df.groupby("name").count().filter(F.col("count")>1).show(truncate = False)

In [ ]:
df.filter(F.col("name") == "State O Maine Big and Tall Solid Microfleece Lounge Pant" ).limit(5).toPandas()

In [ ]:
df.groupby("name").count().filter(F.col("count")>1).count()

In [ ]:
df.filter(F.col("name") == "Alfred Dunner Women's Proportioned Medium Pant" ).limit(5).toPandas()

In [ ]:
df.groupby("brand").count().filter(F.col("count") > 1).show()

In [ ]:
df.select("brand").distinct().count()

In [ ]:
df.filter(F.col("brand") != F.trim(F.col("brand"))).show()

In [ ]:
df.filter(
    (F.col("retail_price") < F.col("cost")) |
    (F.col("retail_price").isNull()) |
    (F.col("retail_price") < 0)
).show()

In [ ]:
df.select("department").distinct().show()

In [ ]:
df_pro.groupby("distribution_center_id").count().orderBy("distribution_center_id", ascending=True).show()

In [ ]:
df.filter(F.col("distribution_center_id").isNull()).show()

In [5]:
table_path = "hdfs://master:9000/data/bronze/users/users.csv"

schema = load_schema_spark("users")

df_user = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv(table_path)

df_user.limit(5).toPandas()

26/07/14 10:25:30 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: id, first_name, last_name, email, age, gender, state, street_address, postal_code, ityc, country, latitude, longitude, traffic_source, created_at
 Schema: id, first_name, last_name, email, age, gender, state, street_address, postal_code, city, country, latitude, longitude, traffic_source, created_at
Expected: city but found: ityc
CSV file: hdfs://master:9000/data/bronze/users/users.csv


,id,first_name,last_name,email,age,gender,state,street_address,postal_code,city,country,latitude,longitude,traffic_source,created_at
0,457,Timothy,Bush,timothybush@example.net,65,M,Acre,87620 Johnson Hills,69917-400,Rio Branco,Brasil,-9.945568,-67.83561,Search,2022-07-19 13:51:00
1,6578,Elizabeth,Martinez,elizabethmartinez@example.com,34,F,Acre,1705 Nielsen Land,69917-400,Rio Branco,Brasil,-9.945568,-67.83561,Search,2023-11-08 18:49:00
2,36280,Christopher,Mendoza,christophermendoza@example.net,13,M,Acre,125 Turner Isle Apt. 264,69917-400,Rio Branco,Brasil,-9.945568,-67.83561,Email,2019-08-24 06:10:00
3,60193,Jimmy,Conner,jimmyconner@example.com,64,M,Acre,0966 Jose Branch Apt. 008,69917-400,Rio Branco,Brasil,-9.945568,-67.83561,Search,2020-02-15 11:26:00
4,64231,Natasha,Wilson,natashawilson@example.net,25,F,Acre,20798 Phillip Trail Apt. 392,69917-400,Rio Branco,Brasil,-9.945568,-67.83561,Search,2020-03-13 06:45:00


In [ ]:
df_user.count()

In [ ]:
df_user.printSchema()

In [9]:
from pyspark.sql import functions as F
df_user.filter(
    F.col("id").isNull() 
).show()

26/07/14 09:26:09 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: id, first_name, last_name, email, age, gender, state, street_address, postal_code, ityc, country, latitude, longitude, traffic_source, created_at
 Schema: id, first_name, last_name, email, age, gender, state, street_address, postal_code, city, country, latitude, longitude, traffic_source, created_at
Expected: city but found: ityc
CSV file: hdfs://master:9000/data/bronze/users/users.csv


+---+----------+---------+-----+---+------+-----+--------------+-----------+----+-------+--------+---------+--------------+----------+
| id|first_name|last_name|email|age|gender|state|street_address|postal_code|city|country|latitude|longitude|traffic_source|created_at|
+---+----------+---------+-----+---+------+-----+--------------+-----------+----+-------+--------+---------+--------------+----------+
+---+----------+---------+-----+---+------+-----+--------------+-----------+----+-------+--------+---------+--------------+----------+



In [ ]:
df_user.groupby("id").count().filter(F.col("count")>1).show()

In [10]:
df_user.filter(
    F.col("first_name") !=
    F.trim(F.col("first_name"))
).show()

26/07/14 09:26:14 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: id, first_name, last_name, email, age, gender, state, street_address, postal_code, ityc, country, latitude, longitude, traffic_source, created_at
 Schema: id, first_name, last_name, email, age, gender, state, street_address, postal_code, city, country, latitude, longitude, traffic_source, created_at
Expected: city but found: ityc
CSV file: hdfs://master:9000/data/bronze/users/users.csv


+---+----------+---------+-----+---+------+-----+--------------+-----------+----+-------+--------+---------+--------------+----------+
| id|first_name|last_name|email|age|gender|state|street_address|postal_code|city|country|latitude|longitude|traffic_source|created_at|
+---+----------+---------+-----+---+------+-----+--------------+-----------+----+-------+--------+---------+--------------+----------+
+---+----------+---------+-----+---+------+-----+--------------+-----------+----+-------+--------+---------+--------------+----------+



In [11]:
df_user.filter(F.col("email").isNull()).show()

26/07/14 09:26:17 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: id, first_name, last_name, email, age, gender, state, street_address, postal_code, ityc, country, latitude, longitude, traffic_source, created_at
 Schema: id, first_name, last_name, email, age, gender, state, street_address, postal_code, city, country, latitude, longitude, traffic_source, created_at
Expected: city but found: ityc
CSV file: hdfs://master:9000/data/bronze/users/users.csv


+---+----------+---------+-----+---+------+-----+--------------+-----------+----+-------+--------+---------+--------------+----------+
| id|first_name|last_name|email|age|gender|state|street_address|postal_code|city|country|latitude|longitude|traffic_source|created_at|
+---+----------+---------+-----+---+------+-----+--------------+-----------+----+-------+--------+---------+--------------+----------+
+---+----------+---------+-----+---+------+-----+--------------+-----------+----+-------+--------+---------+--------------+----------+



In [13]:
df_user.filter(F.col("email") != F.trim(F.col("email"))).count()

0

In [14]:
df_user.filter(F.col("email") != F.lower(F.col("email"))).count()

0

In [17]:
df_user.select(
    F.min("age").alias("min"),
    F.max("age").alias("max")
).show()

+---+---+
|min|max|
+---+---+
| 12| 70|
+---+---+



In [18]:
df_user.filter(F.col("age").isNull()).count()

0

In [20]:
df_user.select("gender").distinct().show()

+------+
|gender|
+------+
|     F|
|     M|
+------+



In [21]:
df_user.filter(F.col("gender").isNull()).count()

0

In [26]:
df_user.select("state").distinct().count()

229

In [25]:
df_user.filter(F.col("email") != F.trim(F.col("email"))).count()

0

In [33]:
df_user.select("city").distinct().show()

26/07/14 10:11:11 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: ityc
 Schema: city
Expected: city but found: ityc
CSV file: hdfs://master:9000/data/bronze/users/users.csv


+--------------------+
|                city|
+--------------------+
|          Prattville|
|        Dos Hermanas|
|   Saint-Genis-Laval|
|          Sallanches|
|        Ludwigsfelde|
|         Santa Paula|
|             Palermo|
|                 Azé|
|             Paterna|
|Daegu Metropolita...|
|             Boreham|
|           Worcester|
|               Bever|
|             Antwerp|
|         Mancha Real|
|         Misawa City|
|               Tempe|
|            Zirndorf|
|           Germering|
|              Corona|
+--------------------+
only showing top 20 rows



In [32]:
df_user.select("city").filter(F.col("city") != F.trim(F.col("email"))).show()

+--------------+
|          city|
+--------------+
|    Rio Branco|
|    Rio Branco|
|    Rio Branco|
|    Rio Branco|
|    Rio Branco|
|    Rio Branco|
|Sena Madureira|
|Sena Madureira|
|Sena Madureira|
|Sena Madureira|
|Sena Madureira|
|Sena Madureira|
|Sena Madureira|
|Sena Madureira|
|Sena Madureira|
|      Tarauacá|
|      Tarauacá|
|      Tarauacá|
|      Tarauacá|
|      Tarauacá|
+--------------+
only showing top 20 rows



26/07/14 10:10:33 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: email, ityc
 Schema: email, city
Expected: city but found: ityc
CSV file: hdfs://master:9000/data/bronze/users/users.csv


In [6]:
df_user.select("country").distinct().show()

+--------------+
|       country|
+--------------+
|       Germany|
|        France|
|       Belgium|
| United States|
|         China|
|         Spain|
|   South Korea|
|         Japan|
|        Brasil|
|        Poland|
|     Australia|
|      Colombia|
|United Kingdom|
|        España|
|   Deutschland|
|       Austria|
+--------------+



In [7]:
df_user.select("traffic_source").distinct().show()

+--------------+
|traffic_source|
+--------------+
|       Organic|
|       Display|
|         Email|
|        Search|
|      Facebook|
+--------------+



In [11]:
table_path = "hdfs://master:9000/data/bronze/orders/orders.csv"

schema = load_schema_spark("orders")

df_ord = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv(table_path)

df_ord.limit(10).toPandas()

,order_id,user_id,status,gender,created_at,returned_at,shipped_at,delivered_at,num_of_item
0,8,5,Cancelled,F,2022-10-20 10:03:00.000000,NaT,NaT,NaT,3
1,60,44,Cancelled,F,2023-01-20 02:12:00.000000,NaT,NaT,NaT,1
2,64,46,Cancelled,F,2021-12-06 09:11:00.000000,NaT,NaT,NaT,1
3,89,65,Cancelled,F,2020-08-13 09:58:00.000000,NaT,NaT,NaT,1
4,102,76,Cancelled,F,2023-01-17 08:17:00.000000,NaT,NaT,NaT,2
5,117,89,Cancelled,F,2023-07-31 13:25:00.000000,NaT,NaT,NaT,1
6,143,118,Cancelled,F,2020-04-21 02:59:00.000000,NaT,NaT,NaT,1
7,153,124,Cancelled,F,2022-07-10 16:42:00.000000,NaT,NaT,NaT,2
8,182,147,Cancelled,F,2024-01-15 10:29:28.317841,NaT,NaT,NaT,3
9,183,148,Cancelled,F,2023-07-04 09:25:00.000000,NaT,NaT,NaT,1


In [14]:
df_ord.count()

125226

In [12]:
df_ord.select("status").distinct().show()

+----------+
|    status|
+----------+
|  Returned|
|   Shipped|
|Processing|
|  Complete|
| Cancelled|
+----------+



In [13]:
df_ord.select("gender").distinct().show()

+------+
|gender|
+------+
|     F|
|     M|
+------+



In [17]:
from pyspark.sql import functions as F
df_user.filter(F.col("returned_at").isNull()).count()

112696

In [18]:
df_user.filter(F.col("shipped_at").isNull()).count()

43765

In [19]:
df_user.filter(F.col("delivered_at").isNull()).count()

81342